In [1]:
import pandas as pd
import numpy as np

rsf_sub = pd.read_csv("rsf_submission.csv")
cox_sub = pd.read_csv("lasso_cox_baseline_submission.csv")

prob_cols = ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]

# event_id 기준으로 정렬/병합
cox_sub = cox_sub.set_index("event_id").reindex(rsf_sub["event_id"].values).reset_index()

rsf_vals = rsf_sub[prob_cols].values
cox_vals = cox_sub[ prob_cols].values

# RSF + Cox
w_rsf = 0.8 #RSF 가중치
blended = w_rsf * rsf_vals + (1 - w_rsf) * cox_vals
blended = np.maximum.accumulate(blended, axis=1)

ensemble_sub = rsf_sub[["event_id"]].copy()
for i, col in enumerate(prob_cols):
    ensemble_sub[col] = blended[:, i]

ensemble_sub.to_csv("ensemble_submission.csv", index=False)
print(f"Saved → ensemble_submission.csv  (RSF {w_rsf:.0%} + Cox {1-w_rsf:.0%})")
ensemble_sub.head()

Saved → ensemble_submission.csv  (RSF 80% + Cox 20%)


,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.002082,0.003957,0.004745,0.009053
1,13353600,0.482205,0.885053,0.930865,0.974513
2,13942327,0.010413,0.019409,0.023088,0.042142
3,16112781,0.697742,0.901079,0.933389,0.980894
4,17132808,0.143930,0.162744,0.170093,0.205011
